# Notebook 04: Logistic Regression

## Purpose

Fit the primary logistic-regression models using one fixed five-fold assignment, generate out-of-fold probabilities, fit full-sample models for interpretation, and estimate coefficient and probability-scale contrasts with paired bootstrap uncertainty.

## Inputs

- Labelled complete-case data from Notebook 02
- Sample and target metadata from Notebooks 01--02

## Outputs

- Fixed cross-validation fold assignments
- Logistic out-of-fold probabilities and fitted model files
- Coefficient, probability-contrast, and bootstrap tables
- Logistic interpretation figures
- `data/processed/logistic_regression_metadata.json`

## Dependencies

Run Notebooks 01--02 first. Notebooks 05--09 reuse the fixed folds and logistic outputs.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## Modelling decisions fixed before inspecting the results

- Random seed: `26`
- Cross-validation folds: `5`
- Stratification variable: `joint_label_code`
- Continuous predictors:
  - age;
  - BMI;
  - income-to-poverty ratio.
- Categorical predictors:
  - sex;
  - race/ethnicity;
  - insurance history.
- Reference categories:
  - Female;
  - Non-Hispanic White;
  - Continuously insured.
- Continuous variables are standardised within each training fold.
- Categorical variables are one-hot encoded with the fixed references above.
- Primary logistic regression:
  - L2 penalty;
  - `C = 1.0`;
  - no class weighting;
  - identical specification for both targets.
- Bootstrap:
  - paired participant-level resampling;
  - the same resampled participants are used for both targets in every replicate;
  - percentile 95% intervals.

The fixed L2 penalty provides numerical stability and ensures that target comparisons are not confounded by target-specific hyperparameter tuning. Coefficients are therefore **penalised estimates**. Their bootstrap intervals describe empirical estimation stability under the fixed model specification; they are not classical unpenalised Wald intervals or p-values.

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 180)


## 2. Configuration

In [ ]:
RANDOM_STATE = 26
N_SPLITS = 5

LOGISTIC_C = 1.0
MAX_ITER = 5000
SOLVER = "lbfgs"

# The default is appropriate for the final analysis. During a quick technical
# test, it can be temporarily overridden before starting Jupyter:
#   Windows PowerShell: $env:LOGISTIC_N_BOOTSTRAP="50"
#   macOS/Linux:        export LOGISTIC_N_BOOTSTRAP=50
N_BOOTSTRAP = int(
    os.environ.get("LOGISTIC_N_BOOTSTRAP", "1000")
)
BOOTSTRAP_PROGRESS_EVERY = max(1, min(50, N_BOOTSTRAP))

CONTINUOUS_PREDICTORS = [
    "age",
    "bmi",
    "income_poverty_ratio",
]

CATEGORICAL_PREDICTORS = [
    "sex",
    "race_ethnicity",
    "insurance_history",
]

PREDICTOR_COLUMNS = (
    CONTINUOUS_PREDICTORS
    + CATEGORICAL_PREDICTORS
)

TARGET_COLUMNS = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]

TARGET_DISPLAY_NAMES = {
    "self_reported_prior_diagnosis": (
        "Prior reported clinician diagnosis"
    ),
    "current_hba1c_ge_6_5": (
        "Current HbA1c at least 6.5%"
    ),
}

TARGET_SHORT_NAMES = {
    "self_reported_prior_diagnosis": "Prior diagnosis",
    "current_hba1c_ge_6_5": "HbA1c ≥ 6.5%",
}

OOF_PROBABILITY_COLUMNS = {
    "self_reported_prior_diagnosis": (
        "logistic_oof_probability_prior_diagnosis"
    ),
    "current_hba1c_ge_6_5": (
        "logistic_oof_probability_hba1c_ge_6_5"
    ),
}

SEX_LEVELS = [
    "Female",
    "Male",
]

RACE_ETHNICITY_LEVELS = [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other or multiracial",
]

INSURANCE_HISTORY_LEVELS = [
    "Continuously insured",
    "Currently insured, past-year gap",
    "Currently uninsured",
]

CATEGORY_LEVELS = {
    "sex": SEX_LEVELS,
    "race_ethnicity": RACE_ETHNICITY_LEVELS,
    "insurance_history": INSURANCE_HISTORY_LEVELS,
}

REFERENCE_CATEGORIES = {
    "sex": "Female",
    "race_ethnicity": "Non-Hispanic White",
    "insurance_history": "Continuously insured",
}

JOINT_LABEL_ORDER = [
    "D0_H0",
    "D0_H1",
    "D1_H0",
    "D1_H1",
]

CONFIDENCE_LEVEL = 0.95
CI_LOWER_QUANTILE = (1 - CONFIDENCE_LEVEL) / 2
CI_UPPER_QUANTILE = 1 - CI_LOWER_QUANTILE

print("Bootstrap replicates:", N_BOOTSTRAP)


## 3. Project paths

In [ ]:
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LABELLED_DATA_PATH = (
    PROCESSED_DIR
    / "nhanes_diabetes_complete_case_labeled.csv"
)

SAMPLE_METADATA_PATH = (
    PROCESSED_DIR / "sample_metadata.json"
)

SAMPLE_LABEL_CHECKPOINT_PATH = (
    PROCESSED_DIR / "sample_and_label_checkpoint.json"
)

FOLD_ASSIGNMENT_PATH = (
    PROCESSED_DIR / "primary_cv_fold_assignments.csv"
)

OOF_PREDICTIONS_PATH = (
    PROCESSED_DIR / "logistic_oof_predictions.csv"
)

LOGISTIC_METADATA_PATH = (
    PROCESSED_DIR / "logistic_regression_metadata.json"
)

if not LABELLED_DATA_PATH.exists():
    raise FileNotFoundError(
        "The labelled complete-case dataset is missing:\n"
        f"{LABELLED_DATA_PATH}\n"
        "Run Notebooks 01–03 before Notebook 04."
    )

print("Project directory:", PROJECT_DIR)
print("Input dataset:", LABELLED_DATA_PATH)


## 4. Load and validate the final analytic sample

The notebook validates against the metadata rather than relying on a hard-coded sample size.

In [ ]:
data = pd.read_csv(LABELLED_DATA_PATH)

sample_metadata = None
if SAMPLE_METADATA_PATH.exists():
    with SAMPLE_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        sample_metadata = json.load(file)

sample_label_checkpoint = None
if SAMPLE_LABEL_CHECKPOINT_PATH.exists():
    with SAMPLE_LABEL_CHECKPOINT_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        sample_label_checkpoint = json.load(file)

required_columns = set(
    [
        "id",
        "joint_label_code",
        "label_group",
        "confirmed_current_pregnancy",
        "primary_sample_eligible",
        "exam_status",
        "phlebotomy_weight",
        "survey_stratum",
        "survey_psu",
    ]
    + PREDICTOR_COLUMNS
    + TARGET_COLUMNS
)

missing_columns = required_columns.difference(data.columns)

if missing_columns:
    raise KeyError(
        "The labelled dataset is missing required columns: "
        f"{sorted(missing_columns)}"
    )

# Deterministic row order ensures that fold creation does not depend on the
# order in which the CSV happened to be saved.
data = (
    data
    .sort_values("id")
    .reset_index(drop=True)
)

integer_like_columns = [
    "id",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "exam_status",
    "survey_stratum",
    "survey_psu",
]

for column in integer_like_columns:
    data[column] = pd.to_numeric(
        data[column],
        errors="raise",
    ).astype("Int64")

for column in CONTINUOUS_PREDICTORS:
    data[column] = pd.to_numeric(
        data[column],
        errors="raise",
    ).astype(float)

for column in CATEGORICAL_PREDICTORS:
    data[column] = data[column].astype("string")

data["joint_label_code"] = (
    data["joint_label_code"].astype("string")
)

if not data["id"].is_unique:
    raise ValueError(
        "Participant IDs are not unique."
    )

required_model_columns = (
    PREDICTOR_COLUMNS
    + TARGET_COLUMNS
    + [
        "joint_label_code",
        "confirmed_current_pregnancy",
        "primary_sample_eligible",
    ]
)

if data[required_model_columns].isna().sum().sum() != 0:
    raise ValueError(
        "The modelling dataset contains missing required values."
    )

if data["confirmed_current_pregnancy"].ne(0).any():
    raise ValueError(
        "The modelling dataset contains a confirmed current pregnancy."
    )

if not data["primary_sample_eligible"].eq(1).all():
    raise ValueError(
        "The modelling dataset contains an ineligible participant."
    )

if not data["exam_status"].eq(2).all():
    raise ValueError(
        "At least one participant did not complete the MEC examination."
    )

if not data["phlebotomy_weight"].gt(0).all():
    raise ValueError(
        "At least one participant has a non-positive phlebotomy weight."
    )

for target in TARGET_COLUMNS:
    observed_values = set(
        data[target].astype(int).unique()
    )

    if not observed_values.issubset({0, 1}):
        raise ValueError(
            f"{target} contains values other than 0 and 1: "
            f"{sorted(observed_values)}"
        )

expected_joint_label = (
    "D"
    + data["self_reported_prior_diagnosis"]
    .astype(str)
    + "_H"
    + data["current_hba1c_ge_6_5"]
    .astype(str)
)

if not expected_joint_label.eq(
    data["joint_label_code"]
).all():
    raise ValueError(
        "At least one joint-label code disagrees with the two targets."
    )

unexpected_joint_labels = set(
    data["joint_label_code"].unique()
).difference(JOINT_LABEL_ORDER)

if unexpected_joint_labels:
    raise ValueError(
        "Unexpected joint-label codes: "
        f"{sorted(unexpected_joint_labels)}"
    )

for variable, allowed_levels in CATEGORY_LEVELS.items():
    observed_levels = set(
        data[variable].dropna().astype(str).unique()
    )
    unexpected_levels = observed_levels.difference(
        allowed_levels
    )

    if unexpected_levels:
        raise ValueError(
            f"{variable} contains unexpected categories: "
            f"{sorted(unexpected_levels)}"
        )

    reference = REFERENCE_CATEGORIES[variable]
    if reference not in observed_levels:
        raise ValueError(
            f"The reference category {reference!r} is absent "
            f"from {variable}."
        )

placeholder_columns = [
    "income_poverty_ratio",
    "bmi",
    "phlebotomy_weight",
]

for column in placeholder_columns:
    placeholder_mask = (
        data[column].notna()
        & data[column].gt(0)
        & data[column].lt(1e-50)
    )

    if placeholder_mask.any():
        raise ValueError(
            f"{column} still contains an SAS/XPT numeric "
            "missing-value placeholder."
        )

if sample_metadata is not None:
    expected_n = int(
        sample_metadata["n_complete_case"]
    )

    if len(data) != expected_n:
        raise ValueError(
            "The modelling sample size does not match "
            "Notebook 01 metadata."
        )

if sample_label_checkpoint is not None:
    expected_n = int(
        sample_label_checkpoint["complete_case_n"]
    )

    if len(data) != expected_n:
        raise ValueError(
            "The modelling sample size does not match "
            "Notebook 02 checkpoint metadata."
        )

joint_label_counts = (
    data["joint_label_code"]
    .value_counts()
    .reindex(JOINT_LABEL_ORDER)
)

if joint_label_counts.min() < N_SPLITS:
    raise ValueError(
        "At least one joint-label group is too small for "
        f"{N_SPLITS}-fold stratification."
    )

analysis_id_hash = hashlib.sha256(
    ",".join(
        data["id"].astype(str).tolist()
    ).encode("utf-8")
).hexdigest()

validation_summary = {
    "analytic_sample_n": int(len(data)),
    "duplicate_ids": int(
        data["id"].duplicated().sum()
    ),
    "missing_required_model_values": int(
        data[required_model_columns]
        .isna()
        .sum()
        .sum()
    ),
    "confirmed_current_pregnancy_n": int(
        data["confirmed_current_pregnancy"].sum()
    ),
    "prior_diagnosis_positive_n": int(
        data["self_reported_prior_diagnosis"].sum()
    ),
    "hba1c_positive_n": int(
        data["current_hba1c_ge_6_5"].sum()
    ),
    "joint_label_counts": {
        label: int(count)
        for label, count in joint_label_counts.items()
    },
    "analysis_id_sha256": analysis_id_hash,
}

print("Validation passed.")
validation_summary


## 5. Document category frequencies and references

In [ ]:
category_documentation_rows = []

for variable in CATEGORICAL_PREDICTORS:
    counts = (
        data[variable]
        .value_counts()
        .reindex(CATEGORY_LEVELS[variable])
    )

    for level, count in counts.items():
        category_documentation_rows.append(
            {
                "variable": variable,
                "level": level,
                "n": int(count),
                "share": float(count / len(data)),
                "is_reference_category": (
                    level
                    == REFERENCE_CATEGORIES[variable]
                ),
            }
        )

category_documentation = pd.DataFrame(
    category_documentation_rows
)

category_documentation.to_csv(
    TABLE_DIR
    / "logistic_category_reference_documentation.csv",
    index=False,
)

category_documentation


## Shared modelling design

## 6. Create or reuse one fixed five-fold assignment

The fold assignment is stratified by the four-category joint target. This helps each fold retain participants from all concordant and discordant groups.

The assignment is saved once and reused unchanged by later primary modelling notebooks.

In [ ]:
def create_fold_assignments(
    modelling_data: pd.DataFrame,
) -> pd.DataFrame:
    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    fold_values = np.zeros(
        len(modelling_data),
        dtype=int,
    )

    for fold_index, (_, validation_indices) in enumerate(
        splitter.split(
            modelling_data[PREDICTOR_COLUMNS],
            modelling_data["joint_label_code"],
        ),
        start=1,
    ):
        fold_values[validation_indices] = fold_index

    assignments = modelling_data[
        [
            "id",
            "joint_label_code",
            *TARGET_COLUMNS,
        ]
    ].copy()

    assignments["cv_fold"] = fold_values

    return assignments


if FOLD_ASSIGNMENT_PATH.exists():
    fold_assignments = pd.read_csv(
        FOLD_ASSIGNMENT_PATH
    )

    required_fold_columns = {
        "id",
        "joint_label_code",
        "self_reported_prior_diagnosis",
        "current_hba1c_ge_6_5",
        "cv_fold",
    }

    missing_fold_columns = (
        required_fold_columns
        .difference(fold_assignments.columns)
    )

    if missing_fold_columns:
        raise KeyError(
            "The saved fold-assignment file is missing columns: "
            f"{sorted(missing_fold_columns)}"
        )

    fold_assignments["id"] = pd.to_numeric(
        fold_assignments["id"],
        errors="raise",
    ).astype("Int64")

    current_ids = set(
        data["id"].astype(int)
    )
    saved_ids = set(
        fold_assignments["id"].astype(int)
    )

    if current_ids != saved_ids:
        raise ValueError(
            "The existing fold assignment belongs to a different "
            "analytic sample. Remove the old fold file only after "
            "confirming that Notebooks 01–03 were intentionally rerun."
        )

    fold_assignments = (
        fold_assignments
        .sort_values("id")
        .reset_index(drop=True)
    )

    current_comparison = data[
        [
            "id",
            "joint_label_code",
            *TARGET_COLUMNS,
        ]
    ].copy()

    current_comparison = (
        current_comparison
        .sort_values("id")
        .reset_index(drop=True)
    )

    for column in [
        "joint_label_code",
        *TARGET_COLUMNS,
    ]:
        if not (
            fold_assignments[column].astype(str)
            .eq(current_comparison[column].astype(str))
            .all()
        ):
            raise ValueError(
                "The saved fold assignments disagree with the "
                f"current data in {column}."
            )

    print(
        "Reused the existing fixed fold assignment:"
    )
    print(FOLD_ASSIGNMENT_PATH)

else:
    fold_assignments = create_fold_assignments(
        data
    )

    fold_assignments.to_csv(
        FOLD_ASSIGNMENT_PATH,
        index=False,
    )

    print("Created and saved the fixed fold assignment:")
    print(FOLD_ASSIGNMENT_PATH)

if set(fold_assignments["cv_fold"]) != set(
    range(1, N_SPLITS + 1)
):
    raise ValueError(
        "The fold assignment does not contain exactly folds 1–5."
    )

if fold_assignments["id"].duplicated().any():
    raise ValueError(
        "At least one participant received more than one fold."
    )

data = data.merge(
    fold_assignments[["id", "cv_fold"]],
    on="id",
    how="left",
    validate="one_to_one",
)

if data["cv_fold"].isna().any():
    raise ValueError(
        "At least one participant did not receive a fold."
    )

data["cv_fold"] = data["cv_fold"].astype(int)


## 7. Inspect fold balance

In [ ]:
fold_balance = (
    data
    .groupby(
        ["cv_fold", "joint_label_code"],
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=range(1, N_SPLITS + 1),
        columns=JOINT_LABEL_ORDER,
        fill_value=0,
    )
)

fold_target_balance_rows = []

for fold in range(1, N_SPLITS + 1):
    fold_data = data.loc[
        data["cv_fold"] == fold
    ]

    row = {
        "cv_fold": fold,
        "fold_n": len(fold_data),
    }

    for target in TARGET_COLUMNS:
        row[f"{target}_positive_n"] = int(
            fold_data[target].sum()
        )
        row[f"{target}_positive_share"] = float(
            fold_data[target].mean()
        )

    fold_target_balance_rows.append(row)

fold_target_balance = pd.DataFrame(
    fold_target_balance_rows
)

if (fold_balance == 0).any().any():
    raise ValueError(
        "At least one fold is missing a joint-label group."
    )

for target in TARGET_COLUMNS:
    if (
        fold_target_balance[
            f"{target}_positive_n"
        ].eq(0).any()
    ):
        raise ValueError(
            f"At least one fold has no positive {target} cases."
        )

fold_balance.to_csv(
    TABLE_DIR / "primary_cv_joint_label_balance.csv"
)

fold_target_balance.to_csv(
    TABLE_DIR / "primary_cv_target_balance.csv",
    index=False,
)

display(fold_balance)
display(fold_target_balance)


## Section 7, Step 1 — Preprocessing

## 8. Build the identical preprocessing and model pipeline

The continuous-variable means and standard deviations are learned only from the training data used in each fit. The categorical level order and reference categories are fixed in advance.

In [ ]:
def build_logistic_pipeline() -> Pipeline:
    categorical_encoder = OneHotEncoder(
        categories=[
            SEX_LEVELS,
            RACE_ETHNICITY_LEVELS,
            INSURANCE_HISTORY_LEVELS,
        ],
        drop=[
            REFERENCE_CATEGORIES["sex"],
            REFERENCE_CATEGORIES[
                "race_ethnicity"
            ],
            REFERENCE_CATEGORIES[
                "insurance_history"
            ],
        ],
        handle_unknown="error",
        sparse_output=False,
        dtype=float,
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "continuous",
                StandardScaler(),
                CONTINUOUS_PREDICTORS,
            ),
            (
                "categorical",
                categorical_encoder,
                CATEGORICAL_PREDICTORS,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )

    classifier = LogisticRegression(
        penalty="l2",
        C=LOGISTIC_C,
        solver=SOLVER,
        max_iter=MAX_ITER,
        class_weight=None,
        fit_intercept=True,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", classifier),
        ]
    )


def fit_logistic_pipeline(
    X: pd.DataFrame,
    y: pd.Series,
) -> Pipeline:
    pipeline = build_logistic_pipeline()

    with warnings.catch_warnings():
        warnings.simplefilter(
            "error",
            ConvergenceWarning,
        )

        pipeline.fit(X, y)

    n_iterations = int(
        pipeline
        .named_steps["model"]
        .n_iter_[0]
    )

    if n_iterations >= MAX_ITER:
        raise RuntimeError(
            "Logistic regression reached MAX_ITER without "
            "a confirmed convergence margin."
        )

    return pipeline


def cleaned_feature_names(
    fitted_pipeline: Pipeline,
) -> list[str]:
    raw_names = (
        fitted_pipeline
        .named_steps["preprocessor"]
        .get_feature_names_out()
    )

    cleaned_names = []

    for raw_name in raw_names:
        cleaned_name = (
            raw_name
            .replace("continuous__", "")
            .replace("categorical__", "")
        )

        cleaned_names.append(cleaned_name)

    return cleaned_names


In [ ]:
# Technical pipeline check using the full predictor matrix.
pipeline_check = fit_logistic_pipeline(
    data[PREDICTOR_COLUMNS],
    data[
        "self_reported_prior_diagnosis"
    ].astype(int),
)

encoded_feature_names = cleaned_feature_names(
    pipeline_check
)

print("Encoded model features:")
for feature_name in encoded_feature_names:
    print("-", feature_name)

print(
    "\nTotal encoded predictors:",
    len(encoded_feature_names),
)


## 9. Encoded-feature dictionary

In [ ]:
TERM_DOCUMENTATION = {
    "age": {
        "display_name": "Age (per 1 SD)",
        "variable": "age",
        "comparison": "One-SD increase",
        "reference": "Observed value",
        "term_type": "continuous_standardised",
    },
    "bmi": {
        "display_name": "BMI (per 1 SD)",
        "variable": "bmi",
        "comparison": "One-SD increase",
        "reference": "Observed value",
        "term_type": "continuous_standardised",
    },
    "income_poverty_ratio": {
        "display_name": (
            "Income-to-poverty ratio (per 1 SD)"
        ),
        "variable": "income_poverty_ratio",
        "comparison": "One-SD increase",
        "reference": "Observed value",
        "term_type": "continuous_standardised",
    },
    "sex_Male": {
        "display_name": "Male vs Female",
        "variable": "sex",
        "comparison": "Male",
        "reference": "Female",
        "term_type": "categorical_indicator",
    },
    "race_ethnicity_Mexican American": {
        "display_name": (
            "Mexican American vs Non-Hispanic White"
        ),
        "variable": "race_ethnicity",
        "comparison": "Mexican American",
        "reference": "Non-Hispanic White",
        "term_type": "categorical_indicator",
    },
    "race_ethnicity_Other Hispanic": {
        "display_name": (
            "Other Hispanic vs Non-Hispanic White"
        ),
        "variable": "race_ethnicity",
        "comparison": "Other Hispanic",
        "reference": "Non-Hispanic White",
        "term_type": "categorical_indicator",
    },
    "race_ethnicity_Non-Hispanic Black": {
        "display_name": (
            "Non-Hispanic Black vs Non-Hispanic White"
        ),
        "variable": "race_ethnicity",
        "comparison": "Non-Hispanic Black",
        "reference": "Non-Hispanic White",
        "term_type": "categorical_indicator",
    },
    "race_ethnicity_Non-Hispanic Asian": {
        "display_name": (
            "Non-Hispanic Asian vs Non-Hispanic White"
        ),
        "variable": "race_ethnicity",
        "comparison": "Non-Hispanic Asian",
        "reference": "Non-Hispanic White",
        "term_type": "categorical_indicator",
    },
    "race_ethnicity_Other or multiracial": {
        "display_name": (
            "Other or multiracial vs Non-Hispanic White"
        ),
        "variable": "race_ethnicity",
        "comparison": "Other or multiracial",
        "reference": "Non-Hispanic White",
        "term_type": "categorical_indicator",
    },
    (
        "insurance_history_"
        "Currently insured, past-year gap"
    ): {
        "display_name": (
            "Currently insured with past-year gap "
            "vs continuously insured"
        ),
        "variable": "insurance_history",
        "comparison": (
            "Currently insured, past-year gap"
        ),
        "reference": "Continuously insured",
        "term_type": "categorical_indicator",
    },
    "insurance_history_Currently uninsured": {
        "display_name": (
            "Currently uninsured vs continuously insured"
        ),
        "variable": "insurance_history",
        "comparison": "Currently uninsured",
        "reference": "Continuously insured",
        "term_type": "categorical_indicator",
    },
}

undocumented_terms = set(
    encoded_feature_names
).difference(TERM_DOCUMENTATION)

if undocumented_terms:
    raise KeyError(
        "The following encoded terms lack documentation: "
        f"{sorted(undocumented_terms)}"
    )

feature_dictionary_rows = []

for term in encoded_feature_names:
    documentation = TERM_DOCUMENTATION[term]

    feature_dictionary_rows.append(
        {
            "encoded_term": term,
            **documentation,
        }
    )

feature_dictionary = pd.DataFrame(
    feature_dictionary_rows
)

feature_dictionary.to_csv(
    TABLE_DIR
    / "logistic_encoded_feature_dictionary.csv",
    index=False,
)

feature_dictionary


## Section 7, Step 2 — Fit the two primary logistic models

## 10. Generate out-of-fold probability predictions

Each participant receives a prediction from a model that was not trained on that participant.

In [ ]:
X_all = data[PREDICTOR_COLUMNS].copy()

oof_predictions = data[
    [
        "id",
        "cv_fold",
        "joint_label_code",
        *TARGET_COLUMNS,
    ]
].copy()

fold_fit_rows = []
fold_coefficient_rows = []
fold_scaling_rows = []

for target in TARGET_COLUMNS:
    probability_column = (
        OOF_PROBABILITY_COLUMNS[target]
    )

    oof_predictions[probability_column] = np.nan

    for fold in range(1, N_SPLITS + 1):
        validation_mask = (
            data["cv_fold"] == fold
        )
        training_mask = ~validation_mask

        X_train = X_all.loc[training_mask]
        X_valid = X_all.loc[validation_mask]

        y_train = (
            data.loc[training_mask, target]
            .astype(int)
        )
        y_valid = (
            data.loc[validation_mask, target]
            .astype(int)
        )

        fitted_pipeline = fit_logistic_pipeline(
            X_train,
            y_train,
        )

        validation_probability = (
            fitted_pipeline
            .predict_proba(X_valid)[:, 1]
        )

        oof_predictions.loc[
            validation_mask,
            probability_column,
        ] = validation_probability

        fitted_classifier = (
            fitted_pipeline
            .named_steps["model"]
        )

        fold_fit_rows.append(
            {
                "target": target,
                "target_display_name": (
                    TARGET_DISPLAY_NAMES[target]
                ),
                "cv_fold": fold,
                "training_n": int(
                    training_mask.sum()
                ),
                "validation_n": int(
                    validation_mask.sum()
                ),
                "training_positive_n": int(
                    y_train.sum()
                ),
                "validation_positive_n": int(
                    y_valid.sum()
                ),
                "training_positive_share": float(
                    y_train.mean()
                ),
                "validation_positive_share": float(
                    y_valid.mean()
                ),
                "n_iterations": int(
                    fitted_classifier.n_iter_[0]
                ),
                "minimum_validation_probability": float(
                    validation_probability.min()
                ),
                "maximum_validation_probability": float(
                    validation_probability.max()
                ),
            }
        )

        feature_names = cleaned_feature_names(
            fitted_pipeline
        )

        for term, estimate in zip(
            feature_names,
            fitted_classifier.coef_[0],
        ):
            fold_coefficient_rows.append(
                {
                    "target": target,
                    "cv_fold": fold,
                    "term": term,
                    "coefficient": float(estimate),
                }
            )

        # Scaling is identical across targets because the same participants
        # are in the same training fold. Save it only once.
        if target == TARGET_COLUMNS[0]:
            scaler = (
                fitted_pipeline
                .named_steps["preprocessor"]
                .named_transformers_[
                    "continuous"
                ]
            )

            for variable, mean_value, scale_value in zip(
                CONTINUOUS_PREDICTORS,
                scaler.mean_,
                scaler.scale_,
            ):
                fold_scaling_rows.append(
                    {
                        "cv_fold": fold,
                        "variable": variable,
                        "training_mean": float(
                            mean_value
                        ),
                        "training_standard_deviation": float(
                            scale_value
                        ),
                    }
                )

for target in TARGET_COLUMNS:
    probability_column = (
        OOF_PROBABILITY_COLUMNS[target]
    )

    if (
        oof_predictions[probability_column]
        .isna()
        .any()
    ):
        raise ValueError(
            f"Missing out-of-fold probabilities for {target}."
        )

    if not (
        oof_predictions[probability_column]
        .between(0, 1)
        .all()
    ):
        raise ValueError(
            f"Out-of-fold probabilities for {target} "
            "lie outside [0, 1]."
        )

fold_fit_summary = pd.DataFrame(
    fold_fit_rows
)

fold_coefficients = pd.DataFrame(
    fold_coefficient_rows
)

fold_scaling = pd.DataFrame(
    fold_scaling_rows
)

oof_predictions.to_csv(
    OOF_PREDICTIONS_PATH,
    index=False,
)

fold_fit_summary.to_csv(
    TABLE_DIR / "logistic_fold_fit_summary.csv",
    index=False,
)

fold_coefficients.to_csv(
    TABLE_DIR / "logistic_fold_coefficients.csv",
    index=False,
)

fold_scaling.to_csv(
    TABLE_DIR / "logistic_fold_scaling.csv",
    index=False,
)

print("Saved out-of-fold predictions to:")
print(OOF_PREDICTIONS_PATH)

fold_fit_summary


Performance metrics are intentionally not calculated here. Notebook 06 will evaluate these out-of-fold probabilities using the same definitions for logistic regression and EBM.

## 11. Fit full-sample models for interpretation

Full-sample models are used for coefficient and probability-scale interpretation. Their fitted probabilities are not substituted for the out-of-fold predictions in performance evaluation.

In [ ]:
full_models = {}
full_scaling_rows = []

for target in TARGET_COLUMNS:
    full_model = fit_logistic_pipeline(
        X_all,
        data[target].astype(int),
    )

    full_models[target] = full_model

    model_path = (
        MODEL_DIR
        / f"logistic_{target}.joblib"
    )

    joblib.dump(
        full_model,
        model_path,
    )

    scaler = (
        full_model
        .named_steps["preprocessor"]
        .named_transformers_["continuous"]
    )

    for variable, mean_value, scale_value in zip(
        CONTINUOUS_PREDICTORS,
        scaler.mean_,
        scaler.scale_,
    ):
        full_scaling_rows.append(
            {
                "target": target,
                "variable": variable,
                "full_sample_mean": float(
                    mean_value
                ),
                "full_sample_standard_deviation": float(
                    scale_value
                ),
            }
        )

full_sample_scaling = pd.DataFrame(
    full_scaling_rows
)

full_sample_scaling.to_csv(
    TABLE_DIR / "logistic_full_sample_scaling.csv",
    index=False,
)

print("Saved full-sample models to:")
for target in TARGET_COLUMNS:
    print(
        "-",
        MODEL_DIR / f"logistic_{target}.joblib",
    )

full_sample_scaling


## Section 7, Step 3 — Coefficient interpretation

## 12. Extract standardised coefficients and odds ratios

Continuous-variable coefficients correspond to a one-standard-deviation increase under the full-sample standardisation. Categorical coefficients compare each displayed category with its prespecified reference category.

In [ ]:
def extract_coefficient_table(
    fitted_pipeline: Pipeline,
    target: str,
) -> pd.DataFrame:
    classifier = (
        fitted_pipeline
        .named_steps["model"]
    )

    feature_names = cleaned_feature_names(
        fitted_pipeline
    )

    rows = [
        {
            "target": target,
            "target_display_name": (
                TARGET_DISPLAY_NAMES[target]
            ),
            "term": "Intercept",
            "display_name": "Intercept",
            "variable": "Intercept",
            "comparison": "",
            "reference": "",
            "term_type": "intercept",
            "coefficient": float(
                classifier.intercept_[0]
            ),
        }
    ]

    for term, estimate in zip(
        feature_names,
        classifier.coef_[0],
    ):
        documentation = TERM_DOCUMENTATION[term]

        rows.append(
            {
                "target": target,
                "target_display_name": (
                    TARGET_DISPLAY_NAMES[target]
                ),
                "term": term,
                **documentation,
                "coefficient": float(estimate),
            }
        )

    result = pd.DataFrame(rows)
    result["odds_ratio"] = np.exp(
        result["coefficient"]
    )

    result["coefficient_direction"] = np.select(
        [
            result["coefficient"] > 0,
            result["coefficient"] < 0,
        ],
        [
            "Positive association with modelled odds",
            "Negative association with modelled odds",
        ],
        default="No estimated association",
    )

    return result


point_coefficient_tables = []

for target, fitted_pipeline in full_models.items():
    point_coefficient_tables.append(
        extract_coefficient_table(
            fitted_pipeline,
            target,
        )
    )

point_coefficients = pd.concat(
    point_coefficient_tables,
    ignore_index=True,
)

point_coefficients.to_csv(
    TABLE_DIR
    / "logistic_full_sample_coefficients_point_estimates.csv",
    index=False,
)

point_coefficients


## Coefficient interpretation rule

Within one target, an odds ratio above one means that the predictor is associated with higher modelled odds of that target, conditional on the other predictors and under the fixed penalised model.

Raw coefficients from the two target models are **not** interpreted as directly comparable measures of clinical importance. Cross-target interpretation is prioritised on the probability scale below.

## Section 7, Step 4 — Probability-scale comparison

## 13. Pre-specify probability effects before calculation

Two kinds of effects are calculated:

1. **Average marginal effects**
   - continuous variables: average derivative of predicted probability for a one-SD increase;
   - categorical variables: average discrete probability change from the reference category to the comparison category.
2. **Predicted-risk contrasts**
   - set one predictor to two prespecified values for every participant;
   - keep all other observed predictors unchanged;
   - average the resulting probability difference.

All effects are associations from the fitted model, not causal effects.

In [ ]:
EFFECT_SPECIFICATIONS = [
    # Continuous average marginal effects: derivative per one SD.
    {
        "effect_id": "ame_age_1sd",
        "effect_type": "average_marginal_effect",
        "predictor": "age",
        "term": "age",
        "display_name": "Age: one-SD increase",
        "calculation": "continuous_derivative",
        "reference": "Observed age",
        "comparison": "One-SD increase",
    },
    {
        "effect_id": "ame_bmi_1sd",
        "effect_type": "average_marginal_effect",
        "predictor": "bmi",
        "term": "bmi",
        "display_name": "BMI: one-SD increase",
        "calculation": "continuous_derivative",
        "reference": "Observed BMI",
        "comparison": "One-SD increase",
    },
    {
        "effect_id": "ame_income_1sd",
        "effect_type": "average_marginal_effect",
        "predictor": "income_poverty_ratio",
        "term": "income_poverty_ratio",
        "display_name": (
            "Income-to-poverty ratio: one-SD increase"
        ),
        "calculation": "continuous_derivative",
        "reference": "Observed income-to-poverty ratio",
        "comparison": "One-SD increase",
    },

    # Categorical average discrete changes.
    {
        "effect_id": "adc_sex_male_vs_female",
        "effect_type": "average_discrete_change",
        "predictor": "sex",
        "display_name": "Male vs Female",
        "calculation": "categorical_change",
        "reference": "Female",
        "comparison": "Male",
    },
    {
        "effect_id": "adc_race_mexican_american",
        "effect_type": "average_discrete_change",
        "predictor": "race_ethnicity",
        "display_name": (
            "Mexican American vs Non-Hispanic White"
        ),
        "calculation": "categorical_change",
        "reference": "Non-Hispanic White",
        "comparison": "Mexican American",
    },
    {
        "effect_id": "adc_race_other_hispanic",
        "effect_type": "average_discrete_change",
        "predictor": "race_ethnicity",
        "display_name": (
            "Other Hispanic vs Non-Hispanic White"
        ),
        "calculation": "categorical_change",
        "reference": "Non-Hispanic White",
        "comparison": "Other Hispanic",
    },
    {
        "effect_id": "adc_race_non_hispanic_black",
        "effect_type": "average_discrete_change",
        "predictor": "race_ethnicity",
        "display_name": (
            "Non-Hispanic Black vs Non-Hispanic White"
        ),
        "calculation": "categorical_change",
        "reference": "Non-Hispanic White",
        "comparison": "Non-Hispanic Black",
    },
    {
        "effect_id": "adc_race_non_hispanic_asian",
        "effect_type": "average_discrete_change",
        "predictor": "race_ethnicity",
        "display_name": (
            "Non-Hispanic Asian vs Non-Hispanic White"
        ),
        "calculation": "categorical_change",
        "reference": "Non-Hispanic White",
        "comparison": "Non-Hispanic Asian",
    },
    {
        "effect_id": "adc_race_other_multiracial",
        "effect_type": "average_discrete_change",
        "predictor": "race_ethnicity",
        "display_name": (
            "Other or multiracial vs Non-Hispanic White"
        ),
        "calculation": "categorical_change",
        "reference": "Non-Hispanic White",
        "comparison": "Other or multiracial",
    },
    {
        "effect_id": "adc_insurance_gap",
        "effect_type": "average_discrete_change",
        "predictor": "insurance_history",
        "display_name": (
            "Currently insured with past-year gap "
            "vs continuously insured"
        ),
        "calculation": "categorical_change",
        "reference": "Continuously insured",
        "comparison": (
            "Currently insured, past-year gap"
        ),
    },
    {
        "effect_id": "adc_currently_uninsured",
        "effect_type": "average_discrete_change",
        "predictor": "insurance_history",
        "display_name": (
            "Currently uninsured vs continuously insured"
        ),
        "calculation": "categorical_change",
        "reference": "Continuously insured",
        "comparison": "Currently uninsured",
    },

    # Prespecified predicted-risk contrasts.
    {
        "effect_id": "contrast_bmi_25_30",
        "effect_type": "predicted_risk_contrast",
        "predictor": "bmi",
        "display_name": "BMI: 25 to 30",
        "calculation": "continuous_contrast",
        "reference": 25.0,
        "comparison": 30.0,
    },
    {
        "effect_id": "contrast_age_40_60",
        "effect_type": "predicted_risk_contrast",
        "predictor": "age",
        "display_name": "Age: 40 to 60 years",
        "calculation": "continuous_contrast",
        "reference": 40.0,
        "comparison": 60.0,
    },
    {
        "effect_id": "contrast_income_1_2",
        "effect_type": "predicted_risk_contrast",
        "predictor": "income_poverty_ratio",
        "display_name": (
            "Income-to-poverty ratio: 1 to 2"
        ),
        "calculation": "continuous_contrast",
        "reference": 1.0,
        "comparison": 2.0,
    },
    {
        "effect_id": (
            "contrast_insurance_continuous_uninsured"
        ),
        "effect_type": "predicted_risk_contrast",
        "predictor": "insurance_history",
        "display_name": (
            "Insurance: continuously insured "
            "to currently uninsured"
        ),
        "calculation": "categorical_change",
        "reference": "Continuously insured",
        "comparison": "Currently uninsured",
    },
]

effect_specification_table = pd.DataFrame(
    EFFECT_SPECIFICATIONS
)

effect_specification_table.to_csv(
    TABLE_DIR
    / "logistic_probability_effect_specifications.csv",
    index=False,
)

effect_specification_table


## 14. Verify that prespecified finite contrasts lie within observed support

In [ ]:
contrast_support_rows = []

for specification in EFFECT_SPECIFICATIONS:
    if (
        specification["calculation"]
        != "continuous_contrast"
    ):
        continue

    variable = specification["predictor"]
    observed_minimum = float(
        data[variable].min()
    )
    observed_maximum = float(
        data[variable].max()
    )

    reference = float(
        specification["reference"]
    )
    comparison = float(
        specification["comparison"]
    )

    within_support = (
        observed_minimum
        <= reference
        <= observed_maximum
        and observed_minimum
        <= comparison
        <= observed_maximum
    )

    contrast_support_rows.append(
        {
            "effect_id": specification["effect_id"],
            "predictor": variable,
            "observed_minimum": observed_minimum,
            "observed_maximum": observed_maximum,
            "reference_value": reference,
            "comparison_value": comparison,
            "both_values_within_observed_range": (
                within_support
            ),
        }
    )

contrast_support = pd.DataFrame(
    contrast_support_rows
)

if not contrast_support[
    "both_values_within_observed_range"
].all():
    raise ValueError(
        "At least one prespecified contrast lies outside "
        "the observed predictor range."
    )

contrast_support.to_csv(
    TABLE_DIR
    / "logistic_probability_contrast_support.csv",
    index=False,
)

contrast_support


## 15. Probability-effect helper functions

In [ ]:
def coefficient_lookup(
    fitted_pipeline: Pipeline,
) -> dict[str, float]:
    classifier = (
        fitted_pipeline
        .named_steps["model"]
    )

    return dict(
        zip(
            cleaned_feature_names(
                fitted_pipeline
            ),
            classifier.coef_[0],
        )
    )


def calculate_probability_effects(
    fitted_pipeline: Pipeline,
    evaluation_data: pd.DataFrame,
    target: str,
) -> pd.DataFrame:
    baseline_probability = (
        fitted_pipeline
        .predict_proba(evaluation_data)[:, 1]
    )

    coefficients = coefficient_lookup(
        fitted_pipeline
    )

    rows = []

    for specification in EFFECT_SPECIFICATIONS:
        calculation = specification[
            "calculation"
        ]
        predictor = specification[
            "predictor"
        ]

        if calculation == "continuous_derivative":
            term = specification["term"]

            # The corresponding predictor has been standardised.
            # A one-unit change on this scale equals a one-SD change
            # in the original predictor.
            effect = float(
                np.mean(
                    baseline_probability
                    * (1 - baseline_probability)
                    * coefficients[term]
                )
            )

        elif calculation in {
            "categorical_change",
            "continuous_contrast",
        }:
            reference_data = evaluation_data.copy()
            comparison_data = evaluation_data.copy()

            reference_data[predictor] = (
                specification["reference"]
            )
            comparison_data[predictor] = (
                specification["comparison"]
            )

            reference_probability = (
                fitted_pipeline
                .predict_proba(reference_data)[:, 1]
            )
            comparison_probability = (
                fitted_pipeline
                .predict_proba(comparison_data)[:, 1]
            )

            effect = float(
                np.mean(
                    comparison_probability
                    - reference_probability
                )
            )

        else:
            raise ValueError(
                f"Unknown effect calculation: {calculation}"
            )

        rows.append(
            {
                "target": target,
                "target_display_name": (
                    TARGET_DISPLAY_NAMES[target]
                ),
                "effect_id": specification[
                    "effect_id"
                ],
                "effect_type": specification[
                    "effect_type"
                ],
                "predictor": predictor,
                "display_name": specification[
                    "display_name"
                ],
                "reference": specification[
                    "reference"
                ],
                "comparison": specification[
                    "comparison"
                ],
                "estimate_probability": effect,
                "estimate_percentage_points": (
                    100 * effect
                ),
                "evaluation_n": int(
                    len(evaluation_data)
                ),
            }
        )

    return pd.DataFrame(rows)


## 16. Calculate full-sample probability effects

In [ ]:
point_effect_tables = []

for target, fitted_pipeline in full_models.items():
    point_effect_tables.append(
        calculate_probability_effects(
            fitted_pipeline,
            X_all,
            target,
        )
    )

point_probability_effects = pd.concat(
    point_effect_tables,
    ignore_index=True,
)

point_probability_effects.to_csv(
    TABLE_DIR
    / "logistic_probability_effects_point_estimates.csv",
    index=False,
)

point_probability_effects


## 17. Participant-level paired bootstrap

Every bootstrap replicate resamples participants once and then fits both target models to that same resample. This preserves the paired structure needed for cross-target differences.

The bootstrap also repeats preprocessing from the resampled data, so uncertainty in scaling and coefficient estimation is propagated.

In [ ]:
def bootstrap_one_replicate(
    replicate: int,
    sample_indices: np.ndarray,
) -> tuple[list[dict], list[dict]]:
    bootstrap_data = (
        data
        .iloc[sample_indices]
        .reset_index(drop=True)
    )

    X_bootstrap = bootstrap_data[
        PREDICTOR_COLUMNS
    ].copy()

    coefficient_rows = []
    effect_rows = []

    for target in TARGET_COLUMNS:
        y_bootstrap = (
            bootstrap_data[target]
            .astype(int)
        )

        if y_bootstrap.nunique() != 2:
            raise RuntimeError(
                f"Bootstrap replicate {replicate} has "
                f"only one class for {target}."
            )

        fitted_pipeline = fit_logistic_pipeline(
            X_bootstrap,
            y_bootstrap,
        )

        coefficient_table = (
            extract_coefficient_table(
                fitted_pipeline,
                target,
            )
        )

        for row in coefficient_table.to_dict(
            orient="records"
        ):
            coefficient_rows.append(
                {
                    "bootstrap_replicate": replicate,
                    "target": target,
                    "term": row["term"],
                    "coefficient": row[
                        "coefficient"
                    ],
                }
            )

        effect_table = calculate_probability_effects(
            fitted_pipeline,
            X_bootstrap,
            target,
        )

        for row in effect_table.to_dict(
            orient="records"
        ):
            effect_rows.append(
                {
                    "bootstrap_replicate": replicate,
                    "target": target,
                    "effect_id": row["effect_id"],
                    "estimate_probability": row[
                        "estimate_probability"
                    ],
                }
            )

    return coefficient_rows, effect_rows


bootstrap_rng = np.random.default_rng(
    RANDOM_STATE
)

bootstrap_coefficient_rows = []
bootstrap_effect_rows = []

bootstrap_start_time = time.time()

for replicate in range(1, N_BOOTSTRAP + 1):
    sample_indices = bootstrap_rng.integers(
        low=0,
        high=len(data),
        size=len(data),
    )

    coefficient_rows, effect_rows = (
        bootstrap_one_replicate(
            replicate=replicate,
            sample_indices=sample_indices,
        )
    )

    bootstrap_coefficient_rows.extend(
        coefficient_rows
    )
    bootstrap_effect_rows.extend(
        effect_rows
    )

    if (
        replicate % BOOTSTRAP_PROGRESS_EVERY == 0
        or replicate == N_BOOTSTRAP
    ):
        elapsed_minutes = (
            time.time() - bootstrap_start_time
        ) / 60

        print(
            f"Completed {replicate:,}/{N_BOOTSTRAP:,} "
            f"bootstrap replicates "
            f"({elapsed_minutes:.1f} minutes elapsed)."
        )

bootstrap_coefficients = pd.DataFrame(
    bootstrap_coefficient_rows
)

bootstrap_probability_effects = pd.DataFrame(
    bootstrap_effect_rows
)

expected_coefficient_rows = (
    N_BOOTSTRAP
    * len(TARGET_COLUMNS)
    * point_coefficients[
        "term"
    ].nunique()
)

expected_effect_rows = (
    N_BOOTSTRAP
    * len(TARGET_COLUMNS)
    * len(EFFECT_SPECIFICATIONS)
)

if len(bootstrap_coefficients) != expected_coefficient_rows:
    raise ValueError(
        "Unexpected number of bootstrap coefficient rows."
    )

if len(bootstrap_probability_effects) != expected_effect_rows:
    raise ValueError(
        "Unexpected number of bootstrap effect rows."
    )

bootstrap_coefficients.to_csv(
    TABLE_DIR
    / "logistic_bootstrap_coefficients_long.csv",
    index=False,
)

bootstrap_probability_effects.to_csv(
    TABLE_DIR
    / "logistic_bootstrap_probability_effects_long.csv",
    index=False,
)

print("Bootstrap completed successfully.")


## 18. Bootstrap coefficient intervals

In [ ]:
coefficient_intervals = (
    bootstrap_coefficients
    .groupby(
        ["target", "term"],
        as_index=False,
    )["coefficient"]
    .agg(
        bootstrap_mean="mean",
        bootstrap_standard_error="std",
        coefficient_ci_lower=(
            lambda values: values.quantile(
                CI_LOWER_QUANTILE
            )
        ),
        coefficient_ci_upper=(
            lambda values: values.quantile(
                CI_UPPER_QUANTILE
            )
        ),
    )
)

coefficient_summary = (
    point_coefficients
    .merge(
        coefficient_intervals,
        on=["target", "term"],
        how="left",
        validate="one_to_one",
    )
)

coefficient_summary[
    "odds_ratio_ci_lower"
] = np.exp(
    coefficient_summary[
        "coefficient_ci_lower"
    ]
)

coefficient_summary[
    "odds_ratio_ci_upper"
] = np.exp(
    coefficient_summary[
        "coefficient_ci_upper"
    ]
)

coefficient_summary[
    "bootstrap_replicates"
] = N_BOOTSTRAP

coefficient_summary.to_csv(
    TABLE_DIR
    / "logistic_coefficients_with_bootstrap_intervals.csv",
    index=False,
)

coefficient_summary


## 19. Bootstrap intervals for target-specific probability effects

In [ ]:
probability_effect_intervals = (
    bootstrap_probability_effects
    .groupby(
        ["target", "effect_id"],
        as_index=False,
    )["estimate_probability"]
    .agg(
        bootstrap_mean_probability="mean",
        bootstrap_standard_error_probability="std",
        ci_lower_probability=(
            lambda values: values.quantile(
                CI_LOWER_QUANTILE
            )
        ),
        ci_upper_probability=(
            lambda values: values.quantile(
                CI_UPPER_QUANTILE
            )
        ),
    )
)

probability_effect_summary = (
    point_probability_effects
    .merge(
        probability_effect_intervals,
        on=["target", "effect_id"],
        how="left",
        validate="one_to_one",
    )
)

probability_effect_summary[
    "ci_lower_percentage_points"
] = (
    100
    * probability_effect_summary[
        "ci_lower_probability"
    ]
)

probability_effect_summary[
    "ci_upper_percentage_points"
] = (
    100
    * probability_effect_summary[
        "ci_upper_probability"
    ]
)

probability_effect_summary[
    "bootstrap_replicates"
] = N_BOOTSTRAP

probability_effect_summary.to_csv(
    TABLE_DIR
    / "logistic_probability_effects_with_bootstrap_intervals.csv",
    index=False,
)

probability_effect_summary


## 20. Paired cross-target differences

The reported difference is

$$
\text{effect for current HbA1c at least 6.5\%}
-
\text{effect for prior reported diagnosis}.
$$

A positive value means that the fitted probability-scale association is larger for the HbA1c target under the fixed model specification. It does not establish greater biological importance or a causal difference.

In [ ]:
point_effect_wide = (
    point_probability_effects
    .pivot(
        index=[
            "effect_id",
            "effect_type",
            "predictor",
            "display_name",
            "reference",
            "comparison",
        ],
        columns="target",
        values="estimate_probability",
    )
    .reset_index()
)

point_effect_wide.columns.name = None

point_effect_wide = point_effect_wide.rename(
    columns={
        "self_reported_prior_diagnosis": (
            "prior_diagnosis_effect_probability"
        ),
        "current_hba1c_ge_6_5": (
            "hba1c_effect_probability"
        ),
    }
)

point_effect_wide[
    "difference_hba1c_minus_prior_probability"
] = (
    point_effect_wide[
        "hba1c_effect_probability"
    ]
    - point_effect_wide[
        "prior_diagnosis_effect_probability"
    ]
)

bootstrap_effect_wide = (
    bootstrap_probability_effects
    .pivot(
        index=[
            "bootstrap_replicate",
            "effect_id",
        ],
        columns="target",
        values="estimate_probability",
    )
    .reset_index()
)

bootstrap_effect_wide.columns.name = None

bootstrap_effect_wide[
    "difference_hba1c_minus_prior_probability"
] = (
    bootstrap_effect_wide[
        "current_hba1c_ge_6_5"
    ]
    - bootstrap_effect_wide[
        "self_reported_prior_diagnosis"
    ]
)

cross_target_intervals = (
    bootstrap_effect_wide
    .groupby(
        "effect_id",
        as_index=False,
    )[
        "difference_hba1c_minus_prior_probability"
    ]
    .agg(
        bootstrap_mean_difference="mean",
        bootstrap_standard_error_difference="std",
        difference_ci_lower_probability=(
            lambda values: values.quantile(
                CI_LOWER_QUANTILE
            )
        ),
        difference_ci_upper_probability=(
            lambda values: values.quantile(
                CI_UPPER_QUANTILE
            )
        ),
    )
)

cross_target_probability_differences = (
    point_effect_wide
    .merge(
        cross_target_intervals,
        on="effect_id",
        how="left",
        validate="one_to_one",
    )
)

for column in [
    "prior_diagnosis_effect_probability",
    "hba1c_effect_probability",
    "difference_hba1c_minus_prior_probability",
    "difference_ci_lower_probability",
    "difference_ci_upper_probability",
]:
    cross_target_probability_differences[
        column.replace(
            "_probability",
            "_percentage_points",
        )
    ] = (
        100
        * cross_target_probability_differences[
            column
        ]
    )

cross_target_probability_differences[
    "bootstrap_replicates"
] = N_BOOTSTRAP

cross_target_probability_differences.to_csv(
    TABLE_DIR
    / "logistic_cross_target_probability_differences.csv",
    index=False,
)

cross_target_probability_differences


## Section 7, Step 5 — Logistic-regression outputs

## 21. Figure 2: selected probability-scale effects

The main figure prioritises effects that are easiest to communicate in a short paper:

- one-SD average marginal effects for age, BMI, and income;
- prespecified finite contrasts for age, BMI, income, and insurance.

All full probability-effect results remain available in the saved table.

In [ ]:
FIGURE_2_EFFECT_ORDER = [
    "ame_age_1sd",
    "ame_bmi_1sd",
    "ame_income_1sd",
    "contrast_age_40_60",
    "contrast_bmi_25_30",
    "contrast_income_1_2",
    "contrast_insurance_continuous_uninsured",
]

figure2_data = (
    probability_effect_summary
    .loc[
        probability_effect_summary[
            "effect_id"
        ].isin(FIGURE_2_EFFECT_ORDER)
    ]
    .copy()
)

figure2_data["effect_id"] = pd.Categorical(
    figure2_data["effect_id"],
    categories=FIGURE_2_EFFECT_ORDER,
    ordered=True,
)

figure2_data = (
    figure2_data
    .sort_values(
        ["effect_id", "target"]
    )
)

display_name_by_effect = (
    figure2_data
    .drop_duplicates("effect_id")
    .set_index("effect_id")["display_name"]
    .to_dict()
)

target_plot_order = TARGET_COLUMNS
vertical_positions = np.arange(
    len(FIGURE_2_EFFECT_ORDER)
)
target_offsets = {
    TARGET_COLUMNS[0]: -0.12,
    TARGET_COLUMNS[1]: 0.12,
}
target_markers = {
    TARGET_COLUMNS[0]: "o",
    TARGET_COLUMNS[1]: "s",
}

figure, axis = plt.subplots(
    figsize=(10, 6)
)

for target in target_plot_order:
    target_data = (
        figure2_data
        .loc[figure2_data["target"] == target]
        .set_index("effect_id")
        .reindex(FIGURE_2_EFFECT_ORDER)
    )

    estimates = target_data[
        "estimate_percentage_points"
    ].to_numpy()

    interval_lower = target_data[
        "ci_lower_percentage_points"
    ].to_numpy()

    interval_upper = target_data[
        "ci_upper_percentage_points"
    ].to_numpy()

    target_positions = (
        vertical_positions
        + target_offsets[target]
    )

    plotted_points = axis.plot(
        estimates,
        target_positions,
        linestyle="none",
        marker=target_markers[target],
        label=TARGET_SHORT_NAMES[target],
    )

    automatic_colour = (
        plotted_points[0].get_color()
    )

    axis.hlines(
        target_positions,
        interval_lower,
        interval_upper,
        color=automatic_colour,
    )

axis.axvline(
    0,
    linewidth=1,
    linestyle="--",
)

axis.set_yticks(
    vertical_positions,
    labels=[
        display_name_by_effect[
            effect_id
        ]
        for effect_id in FIGURE_2_EFFECT_ORDER
    ],
)

axis.set_xlabel(
    "Average change in predicted probability "
    "(percentage points)"
)

axis.set_ylabel("Probability-scale effect")
axis.set_title(
    "Figure 2. Logistic-regression probability-scale effects"
)
axis.legend()
axis.invert_yaxis()

figure.tight_layout()

figure2_png_path = (
    FIGURE_DIR
    / "figure2_logistic_probability_effects.png"
)

figure2_pdf_path = (
    FIGURE_DIR
    / "figure2_logistic_probability_effects.pdf"
)

figure.savefig(
    figure2_png_path,
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure2_pdf_path,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("Saved:")
print(figure2_png_path)
print(figure2_pdf_path)


## 22. Appendix coefficient and odds-ratio forest plot

In [ ]:
forest_data = (
    coefficient_summary
    .loc[
        coefficient_summary["term"]
        != "Intercept"
    ]
    .copy()
)

term_order = encoded_feature_names

forest_data["term"] = pd.Categorical(
    forest_data["term"],
    categories=term_order,
    ordered=True,
)

forest_data = forest_data.sort_values(
    ["term", "target"]
)

display_name_by_term = (
    forest_data
    .drop_duplicates("term")
    .set_index("term")["display_name"]
    .to_dict()
)

vertical_positions = np.arange(
    len(term_order)
)

target_offsets = {
    TARGET_COLUMNS[0]: -0.12,
    TARGET_COLUMNS[1]: 0.12,
}

figure, axis = plt.subplots(
    figsize=(11, 8)
)

for target in TARGET_COLUMNS:
    target_data = (
        forest_data
        .loc[forest_data["target"] == target]
        .set_index("term")
        .reindex(term_order)
    )

    estimates = target_data[
        "odds_ratio"
    ].to_numpy()

    interval_lower = target_data[
        "odds_ratio_ci_lower"
    ].to_numpy()

    interval_upper = target_data[
        "odds_ratio_ci_upper"
    ].to_numpy()

    target_positions = (
        vertical_positions
        + target_offsets[target]
    )

    plotted_points = axis.plot(
        estimates,
        target_positions,
        linestyle="none",
        marker=target_markers[target],
        label=TARGET_SHORT_NAMES[target],
    )

    automatic_colour = (
        plotted_points[0].get_color()
    )

    axis.hlines(
        target_positions,
        interval_lower,
        interval_upper,
        color=automatic_colour,
    )

axis.axvline(
    1,
    linewidth=1,
    linestyle="--",
)

axis.set_xscale("log")

axis.set_yticks(
    vertical_positions,
    labels=[
        display_name_by_term[term]
        for term in term_order
    ],
)

axis.set_xlabel(
    "Penalised odds ratio "
    "(logarithmic axis)"
)

axis.set_ylabel("Model term")
axis.set_title(
    "Appendix. Logistic-regression odds ratios"
)
axis.legend()
axis.invert_yaxis()

figure.tight_layout()

coefficient_forest_png_path = (
    FIGURE_DIR
    / "appendix_logistic_odds_ratio_forest.png"
)

coefficient_forest_pdf_path = (
    FIGURE_DIR
    / "appendix_logistic_odds_ratio_forest.pdf"
)

figure.savefig(
    coefficient_forest_png_path,
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    coefficient_forest_pdf_path,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("Saved:")
print(coefficient_forest_png_path)
print(coefficient_forest_pdf_path)


## 23. Appendix cross-target difference plot

In [ ]:
difference_plot_data = (
    cross_target_probability_differences
    .loc[
        cross_target_probability_differences[
            "effect_id"
        ].isin(FIGURE_2_EFFECT_ORDER)
    ]
    .copy()
)

difference_plot_data["effect_id"] = pd.Categorical(
    difference_plot_data["effect_id"],
    categories=FIGURE_2_EFFECT_ORDER,
    ordered=True,
)

difference_plot_data = (
    difference_plot_data
    .sort_values("effect_id")
)

estimates = difference_plot_data[
    "difference_hba1c_minus_prior_percentage_points"
].to_numpy()

interval_lower = difference_plot_data[
    "difference_ci_lower_percentage_points"
].to_numpy()

interval_upper = difference_plot_data[
    "difference_ci_upper_percentage_points"
].to_numpy()

vertical_positions = np.arange(
    len(difference_plot_data)
)

figure, axis = plt.subplots(
    figsize=(10, 6)
)

plotted_points = axis.plot(
    estimates,
    vertical_positions,
    linestyle="none",
    marker="o",
)

automatic_colour = plotted_points[0].get_color()

axis.hlines(
    vertical_positions,
    interval_lower,
    interval_upper,
    color=automatic_colour,
)

axis.axvline(
    0,
    linewidth=1,
    linestyle="--",
)

axis.set_yticks(
    np.arange(len(difference_plot_data)),
    labels=difference_plot_data[
        "display_name"
    ],
)

axis.set_xlabel(
    "HbA1c-target effect minus prior-diagnosis-target effect "
    "(percentage points)"
)

axis.set_ylabel("Probability-scale effect")
axis.set_title(
    "Appendix. Cross-target differences in logistic effects"
)
axis.invert_yaxis()

figure.tight_layout()

difference_plot_png_path = (
    FIGURE_DIR
    / "appendix_logistic_cross_target_effect_differences.png"
)

difference_plot_pdf_path = (
    FIGURE_DIR
    / "appendix_logistic_cross_target_effect_differences.pdf"
)

figure.savefig(
    difference_plot_png_path,
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    difference_plot_pdf_path,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("Saved:")
print(difference_plot_png_path)
print(difference_plot_pdf_path)


## 24. Interpretation tables for writing

These compact tables are convenient for drafting the results section. The complete machine-readable tables saved above remain the authoritative outputs.

In [ ]:
paper_probability_effects = (
    probability_effect_summary
    .loc[
        probability_effect_summary[
            "effect_id"
        ].isin(FIGURE_2_EFFECT_ORDER),
        [
            "target_display_name",
            "effect_id",
            "display_name",
            "estimate_percentage_points",
            "ci_lower_percentage_points",
            "ci_upper_percentage_points",
        ],
    ]
    .copy()
)

paper_probability_effects[
    "paper_summary"
] = (
    paper_probability_effects[
        "estimate_percentage_points"
    ].map(lambda value: f"{value:.2f}")
    + " percentage points ["
    + paper_probability_effects[
        "ci_lower_percentage_points"
    ].map(lambda value: f"{value:.2f}")
    + ", "
    + paper_probability_effects[
        "ci_upper_percentage_points"
    ].map(lambda value: f"{value:.2f}")
    + "]"
)

paper_cross_target_differences = (
    cross_target_probability_differences
    .loc[
        cross_target_probability_differences[
            "effect_id"
        ].isin(FIGURE_2_EFFECT_ORDER),
        [
            "effect_id",
            "display_name",
            (
                "difference_hba1c_minus_prior_"
                "percentage_points"
            ),
            "difference_ci_lower_percentage_points",
            "difference_ci_upper_percentage_points",
        ],
    ]
    .copy()
)

paper_cross_target_differences[
    "paper_summary"
] = (
    paper_cross_target_differences[
        "difference_hba1c_minus_prior_percentage_points"
    ].map(lambda value: f"{value:.2f}")
    + " percentage points ["
    + paper_cross_target_differences[
        "difference_ci_lower_percentage_points"
    ].map(lambda value: f"{value:.2f}")
    + ", "
    + paper_cross_target_differences[
        "difference_ci_upper_percentage_points"
    ].map(lambda value: f"{value:.2f}")
    + "]"
)

paper_probability_effects.to_csv(
    TABLE_DIR
    / "paper_logistic_probability_effects.csv",
    index=False,
)

paper_cross_target_differences.to_csv(
    TABLE_DIR
    / "paper_logistic_cross_target_differences.csv",
    index=False,
)

display(paper_probability_effects)
display(paper_cross_target_differences)


## Save metadata and final checkpoint

In [ ]:
output_files = {
    "fold_assignments": FOLD_ASSIGNMENT_PATH,
    "oof_predictions": OOF_PREDICTIONS_PATH,
    "category_reference_documentation": (
        TABLE_DIR
        / "logistic_category_reference_documentation.csv"
    ),
    "encoded_feature_dictionary": (
        TABLE_DIR
        / "logistic_encoded_feature_dictionary.csv"
    ),
    "fold_fit_summary": (
        TABLE_DIR / "logistic_fold_fit_summary.csv"
    ),
    "fold_coefficients": (
        TABLE_DIR / "logistic_fold_coefficients.csv"
    ),
    "fold_scaling": (
        TABLE_DIR / "logistic_fold_scaling.csv"
    ),
    "full_sample_scaling": (
        TABLE_DIR
        / "logistic_full_sample_scaling.csv"
    ),
    "coefficient_summary": (
        TABLE_DIR
        / "logistic_coefficients_with_bootstrap_intervals.csv"
    ),
    "probability_effect_summary": (
        TABLE_DIR
        / "logistic_probability_effects_with_bootstrap_intervals.csv"
    ),
    "cross_target_probability_differences": (
        TABLE_DIR
        / "logistic_cross_target_probability_differences.csv"
    ),
    "figure2_png": figure2_png_path,
    "figure2_pdf": figure2_pdf_path,
    "coefficient_forest_png": coefficient_forest_png_path,
    "coefficient_forest_pdf": coefficient_forest_pdf_path,
    "difference_plot_png": difference_plot_png_path,
    "difference_plot_pdf": difference_plot_pdf_path,
    "prior_diagnosis_model": (
        MODEL_DIR
        / "logistic_self_reported_prior_diagnosis.joblib"
    ),
    "hba1c_model": (
        MODEL_DIR
        / "logistic_current_hba1c_ge_6_5.joblib"
    ),
}

missing_output_files = [
    str(path)
    for path in output_files.values()
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "The following expected outputs are missing:\n"
        + "\n".join(missing_output_files)
    )

logistic_metadata = {
    "analytic_sample_n": int(len(data)),
    "analysis_id_sha256": analysis_id_hash,
    "random_state": RANDOM_STATE,
    "n_splits": N_SPLITS,
    "stratification_variable": "joint_label_code",
    "fold_assignment_path": str(
        FOLD_ASSIGNMENT_PATH
    ),
    "predictor_columns": PREDICTOR_COLUMNS,
    "continuous_predictors": CONTINUOUS_PREDICTORS,
    "categorical_predictors": CATEGORICAL_PREDICTORS,
    "reference_categories": REFERENCE_CATEGORIES,
    "target_columns": TARGET_COLUMNS,
    "oof_probability_columns": OOF_PROBABILITY_COLUMNS,
    "logistic_specification": {
        "penalty": "l2",
        "C": LOGISTIC_C,
        "solver": SOLVER,
        "max_iter": MAX_ITER,
        "class_weight": None,
        "continuous_scaling": (
            "StandardScaler fitted within each training fold"
        ),
        "categorical_encoding": (
            "One-hot encoding with fixed reference categories"
        ),
    },
    "bootstrap": {
        "replicates": N_BOOTSTRAP,
        "resampling_unit": "participant row",
        "paired_across_targets": True,
        "confidence_level": CONFIDENCE_LEVEL,
        "interval_method": "percentile",
    },
    "probability_effects": {
        "continuous_ame": (
            "Average derivative of predicted probability "
            "for a one-SD increase"
        ),
        "categorical_effect": (
            "Average discrete probability change from "
            "reference to comparison"
        ),
        "finite_contrasts": [
            {
                "effect_id": specification["effect_id"],
                "predictor": specification["predictor"],
                "reference": specification["reference"],
                "comparison": specification["comparison"],
            }
            for specification in EFFECT_SPECIFICATIONS
            if specification["effect_type"]
            == "predicted_risk_contrast"
        ],
        "cross_target_difference": (
            "HbA1c-target effect minus prior-diagnosis-target effect"
        ),
    },
    "software_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "output_files": {
        name: str(path)
        for name, path in output_files.items()
    },
}

with LOGISTIC_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        logistic_metadata,
        file,
        indent=2,
    )

final_checkpoint = {
    "analytic_sample_n": int(len(data)),
    "participants_with_exactly_one_fold": bool(
        data["cv_fold"].notna().all()
        and data["id"].is_unique
    ),
    "five_folds_present": bool(
        set(data["cv_fold"])
        == set(range(1, N_SPLITS + 1))
    ),
    "all_joint_groups_present_in_each_fold": bool(
        (fold_balance > 0).all().all()
    ),
    "missing_oof_probabilities": int(
        oof_predictions[
            list(
                OOF_PROBABILITY_COLUMNS.values()
            )
        ]
        .isna()
        .sum()
        .sum()
    ),
    "oof_probabilities_within_unit_interval": bool(
        all(
            oof_predictions[column]
            .between(0, 1)
            .all()
            for column in OOF_PROBABILITY_COLUMNS.values()
        )
    ),
    "full_models_saved": bool(
        output_files["prior_diagnosis_model"].exists()
        and output_files["hba1c_model"].exists()
    ),
    "bootstrap_replicates_completed": int(
        bootstrap_probability_effects[
            "bootstrap_replicate"
        ].nunique()
    ),
    "coefficient_terms_per_target": int(
        point_coefficients["term"].nunique()
    ),
    "probability_effects_per_target": int(
        point_probability_effects[
            "effect_id"
        ].nunique()
    ),
    "all_expected_outputs_saved": (
        len(missing_output_files) == 0
    ),
    "metadata_saved": (
        LOGISTIC_METADATA_PATH.exists()
    ),
}

print("Saved logistic-regression metadata to:")
print(LOGISTIC_METADATA_PATH)
print()
print("Final checkpoint:")
final_checkpoint


## Completion criteria

- The same participants, predictors, folds, and preprocessing are used for both targets.
- Out-of-fold predictions are complete and remain separate from full-sample interpretation.
- All model, table, figure, and metadata outputs are written.